In [1]:
# Train a Model on S01-S07 dataset 
# Use the model to infer the trial class on calibration dataset - Get the cleaner trials 
# Retrain or Fine-tune the model using the selected trials only. 
# Verify the performance on the online sessoin data


In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import direction_utils as utils
import direction_learning_utils as train_utils

torch.manual_seed(0)
torch.cuda.manual_seed(0) 
np.random.seed(0)

In [2]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)  # Output size: (batch_size, in_channels, 1, 1)
        
        # self.fc1 = nn.Linear(in_channels, max(1, in_channels // reduction), bias=False)  # Squeeze
        self.fc1 = nn.Linear(in_channels, in_channels // reduction, bias=False)  # Squeeze

        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels, bias=False)  # Excitation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        
        # Squeeze: Global Average Pooling
        out = self.global_avg_pool(x).view(batch_size, channels)  # Shape: (batch_size, in_channels)
        
        # Excitation: Fully connected layers
        out = self.fc1(out)  # Shape: (batch_size, in_channels // reduction)
        out = self.relu(out)
        out = self.fc2(out)  # Shape: (batch_size, in_channels)
        out = self.sigmoid(out).view(batch_size, channels, 1, 1)  # Reshape to (batch_size, in_channels, 1, 1)
        
        # Scale the input by the SE weights
        return x * out.expand_as(x)
    
class SEBlockPerChannel(nn.Module):
    def __init__(self, height, width, reduction=16):
        super(SEBlockPerChannel, self).__init__()
        self.height = height
        self.width = width
        
        # Fully connected layers to learn the importance of each height (electrode) for each channel
        self.fc1 = nn.Linear(height, height // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(height // reduction, height, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Input x shape: (batch_size, channels, height, width)
        b, c, h, w = x.size()
        
        # Initialize a list to store the reweighted channels
        output = []
        # print(f'Shape of X Before SE: {x.size()}')

        # Loop over each channel (for each filter)
        for i in range(c):
            # Extract the i-th channel (filter): shape (batch_size, height, width)
            y = x[:, i, :, :]
            # print(f'Shape of X for filter {i} SE: {y.size()}')

            # Squeeze: Average pooling along the width (temporal dimension)
            y_squeezed = y.mean(dim=-1)  # shape: (batch_size, height)
            # print(f"Shape before FC1 (y_squeezed): {y_squeezed.shape}")
            # print(f'Shape of X for filter {i} Pooling Layer SE: {y_squeezed.size()}')
            
            # Excitation: Fully connected layers to assign weights to the height dimension
            y_fc1 = self.fc1(y_squeezed)
            # print(f'Shape of X for filter {i} FC Layer1 SE: {y_fc1.size()}')
            y_relu = self.relu(y_fc1)

            y_fc2 = self.fc2(y_relu)
            # print(f'Shape of X for filter {i} FC Layer2 SE: {y_fc2.size()}')
            y_sigmoid = self.sigmoid(y_fc2)  # shape: (batch_size, height)
            
            # Reshape to (batch_size, 1, height, 1) to broadcast
            y_sigmoid = y_sigmoid.view(b, 1, h, 1)
            # print(f'Shape of X for filter {i} Sigmoid SE: {y_sigmoid.size()}')
            
            # Scale the original channel data
            y_scaled = y.unsqueeze(1) * y_sigmoid.expand_as(y.unsqueeze(1))  # Reshape y to (batch_size, 1, height, width)
            # print(f'Shape of X for filter {i} After SE: {y_scaled.size()}')
            
            # Append to output list
            output.append(y_scaled)
        
        # Concatenate along the channel dimension to restore the original shape
        output = torch.cat(output, dim=1)  # shape: (batch_size, channels, height, width)
        # print(f'Shape of X for After SE: {output.size()}')
        
        return output


class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        # Squeeze-and-Excitation Block
        self.se_electrode1 = SEBlockPerChannel(Chans, Samples, reduction=3)
        self.se_electrode2 = SEBlockPerChannel(Chans, Samples, reduction=3)
        self.se1 = SEBlock(F1, 3)


        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.se2 = SEBlock(F1*D, 3)  # Squeeze-and-Excitation Block
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.se3 = SEBlock(F2, 3)  # Squeeze-and-Excitation Block
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.se_electrode1(x)
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.se_electrode2(x)
        x = self.se1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.se2(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.se3(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [3]:
def model_using_calibration_data(verbose=False):
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    batch_size=32
    fs = 500
    train_ratio = 0.9
    sub = 7 #Till subject 7 (starting from 0), only calibration sessions are conducted

    # Xtr, Ytr = create_dataset(sub, base_path=parent_dir)
    Xtr, Ytr = utils.calib_sess_dataset(base_path=parent_dir)
    # Creating train-validation split

    eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)
    
    train_loader = train_utils.convert_to_tensor(eeg_train, label_train)
    val_loader = train_utils.convert_to_tensor(eeg_val, label_val)

    

    # X_train = utils.baseline_correction(Xtr)
    # X_train = utils.bandpass_filtering(X_train)

    # X_train_tensor = torch.tensor(eeg_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    # Y_train_tensor = torch.tensor(label_train, dtype=torch.long).to(device)  # Use long for classification

    # X_val_tensor = torch.tensor(eeg_val, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    # Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

    # train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    # test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

    # train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    # val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    # optimizer = optim.Adam(model.parameters(), lr=1e-3)
    trained_model = train_utils.model_training(model, train_loader, val_loader, verbose=verbose)
    torch.save(trained_model.state_dict(), 'calibrated_EEGNetSEPerElectrode_model.pth')  # Save best model

    return trained_model
    

In [4]:
model = model_using_calibration_data(verbose=True)

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\modules\conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,


Epoch 1/1000, Train Loss: 0.6972, Val Loss: 0.6951, Val Acc: 46.5461%
Epoch 2/1000, Train Loss: 0.6852, Val Loss: 0.6967, Val Acc: 42.3520%
Epoch 3/1000, Train Loss: 0.6894, Val Loss: 0.6984, Val Acc: 45.4770%
Epoch 4/1000, Train Loss: 0.6951, Val Loss: 0.6992, Val Acc: 48.1086%
Epoch 5/1000, Train Loss: 0.6863, Val Loss: 0.6995, Val Acc: 46.5461%
Epoch 6/1000, Train Loss: 0.6782, Val Loss: 0.6990, Val Acc: 49.6711%
Epoch 7/1000, Train Loss: 0.6788, Val Loss: 0.6987, Val Acc: 49.6711%
Epoch 8/1000, Train Loss: 0.6790, Val Loss: 0.6982, Val Acc: 49.6711%
Epoch 9/1000, Train Loss: 0.6750, Val Loss: 0.6982, Val Acc: 51.2336%
Epoch 10/1000, Train Loss: 0.6780, Val Loss: 0.6984, Val Acc: 53.8651%
Epoch 11/1000, Train Loss: 0.6734, Val Loss: 0.6979, Val Acc: 53.8651%
Epoch 12/1000, Train Loss: 0.6698, Val Loss: 0.6981, Val Acc: 53.8651%
Epoch 13/1000, Train Loss: 0.6683, Val Loss: 0.6982, Val Acc: 53.8651%
Epoch 14/1000, Train Loss: 0.6672, Val Loss: 0.6983, Val Acc: 56.9901%
Epoch 15/1000, 

In [28]:
# Trial Numbers that achieve performance greater than 70% on the trained model
def trial_selection_from_calibration(model, Xtr, Ytr):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 4
    threshold = 0.6

    data_loader = train_utils.convert_to_tensor(Xtr, Ytr, batch_size=batch_size, shuffle=False)
    model.eval()
    all_trial_indices = []  # To accumulate trial indices across batches

    with torch.no_grad():
        batch = 0
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            
            # Calculate softmax probabilities
            probs = F.softmax(outputs, dim=1)

            # Find indices where probability > threshold
            high_conf_indices = (probs > threshold).nonzero(as_tuple=False)[:, 0]
            
            # Adjust trial indices for the current batch using list comprehension
            adjusted_indices = [(batch * batch_size) + idx.item() for idx in high_conf_indices]

            # Accumulate the adjusted indices
            all_trial_indices.extend(adjusted_indices)
            
            # print(f'Batch {batch} trial indices with score > 70%: {adjusted_indices}')
            
            # Increment batch counter
            batch += 1
    # After the loop, 'all_trial_indices' will contain all the trials that had a score above 70%
    print(f'All trial indices with score > {threshold*100}%: {len(all_trial_indices)}')
    return all_trial_indices

In [29]:
# Model Fine Tuning
def model_fine_tuning_params(model_file, seLayer = False, denseLayer=True, conv2dLayer=False):

    torch.manual_seed(0)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    # for param in model.parameters():
    #         param.requires_grad = False

    # # Unfreeze the last layer parameters
    # for param in model.se_electrode1.parameters():
    #     param.requires_grad = True  # Unfreeze last layer
    
    # # Unfreeze the last layer parameters
    # for param in model.se1.parameters():
    #     param.requires_grad = True  # Unfreeze last layer
    
    # # Unfreeze the last layer parameters
    # for param in model.se2.parameters():
    #     param.requires_grad = True  # Unfreeze last layer
    
    # # Unfreeze the last layer parameters
    # for param in model.se3.parameters():
    #     param.requires_grad = True  # Unfreeze last layer


    if denseLayer or conv2dLayer or seLayer:

        if seLayer:
            print(f'SE layer is fine-tuned')
            
            for param in model.parameters():
                param.requires_grad = False

            # Unfreeze the last layer parameters
            for param in model.se_electrode1.parameters():
                param.requires_grad = True  # Unfreeze last layer
            
            # Unfreeze the last layer parameters
            for param in model.se_electrode2.parameters():
                param.requires_grad = True  # Unfreeze last layer

            # # Unfreeze the last layer parameters
            # for param in model.se1.parameters():
            #     param.requires_grad = True  # Unfreeze last layer
        
            # # Unfreeze the last layer parameters
            # for param in model.se2.parameters():
            #     param.requires_grad = True  # Unfreeze last layer
            
            # # Unfreeze the last layer parameters
            # for param in model.se3.parameters():
            #     param.requires_grad = True  # Unfreeze last layer

        if denseLayer:
            print(f'Dense layer is fine-tuned')
            # Freeze all layers except the last one
            for param in model.dense.parameters():
                param.requires_grad = True  # Freeze all parameters

        if conv2dLayer:
            print(f'Conv2D layer is fine-tuned')
            # Unfreeze the last layer parameters
            for param in model.separableConv.parameters():
                param.requires_grad = True  # Unfreeze last layer

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    return model, optimizer


In [36]:
def subject_specific_fine_tuning(seLayer = False, denseLayer=True, conv2dLayer=False):
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    Xcalib, Ycalib = utils.calib_sess_dataset(base_path=parent_dir)

    train_ratio = 0.9
    batch_size=32
    online_perf = dict()

    model, optimizer = model_fine_tuning_params('calibrated_EEGNetSEPerElectrode_model', seLayer=seLayer, denseLayer=denseLayer, conv2dLayer=conv2dLayer)

    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

        selected_trials = trial_selection_from_calibration(model, Xtr, Ytr)
        if len(selected_trials) < 10:
            continue
        Xtr = Xtr[selected_trials, :, :]
        Ytr = Ytr[selected_trials]

        eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)
        
        # Xtrain = np.concatenate((Xcalib, eeg_train), axis=0)
        # Ytrain = np.concatenate((Ycalib, label_train), axis=0)
        Xtrain = eeg_train
        Ytrain = label_train

        try:
            train_loader = train_utils.convert_to_tensor(Xtrain, Ytrain)
            val_loader = train_utils.convert_to_tensor(eeg_val, label_val)

            trained_model = train_utils.model_training(model, train_loader, val_loader, Tuning=True)

            test_loader = train_utils.convert_to_tensor(Xte, Yte)
            test_loss, test_acc = train_utils.model_evaluation(trained_model, test_loader)

            # model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
            # model.load_state_dict(torch.load('trained_model_checkpoint.pth'))
        except Exception as e:
            print(str(e))
            continue

        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')
        online_perf[f'Sub{sub:02d}'] = test_acc

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))


In [37]:
def evaluation_on_online_data(model_file):
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size=32

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    online_perf = dict()
    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

        test_loader = train_utils.convert_to_tensor(Xte, Yte)

        test_loss, test_acc = train_utils.model_evaluation(model, test_loader)
        online_perf[f'Sub{sub:02d}'] = test_acc
        
        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))

    return

# evaluation_on_online_data('calibrated_EEGNetSEPerElectrode_model')

In [38]:
subject_specific_fine_tuning(seLayer=True, denseLayer=False, conv2dLayer=False)

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
C:\Users\postd\AppData\Local\Temp\ipykernel_10088\3244971408.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting 

SE layer is fine-tuned
All trial indices with score > 60.0%: 42
Subject: 08, Test Accuracy: 51.56%
All trial indices with score > 60.0%: 67
Subject: 09, Test Accuracy: 48.44%
All trial indices with score > 60.0%: 26
Subject: 10, Test Accuracy: 45.31%
All trial indices with score > 60.0%: 60
Subject: 11, Test Accuracy: 54.69%
All trial indices with score > 60.0%: 9
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
All trial indices with score > 60.0%: 0
Average Accuracy: 50.0
[51.5625, 48.4375, 45.3125, 54.6875]
